In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


In [5]:
df = pd.read_csv("../data/processed/dataset_desbalanceado.csv")

target = 'Appointment Type'

num_cols = [
    'Age', 'Number of Diseases', 'Recent Hospitalization',
    'Number of Medications', 'Hour',
    'Creation to Assignment Interval',
    'Number of Previous Attendance',
    'Number of Previous Non-Attendance'
]

cat_cols = ['Sex', 'Insurance Type', 'Day', 'Month']

X = df[num_cols + cat_cols]
y = df[target]

# Preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', 'passthrough', cat_cols)
    ]
)

# Decision Tree

In [7]:
df = pd.read_csv("../data/processed/dataset_desbalanceado.csv")

target = 'Appointment Type'

num_cols = [
    'Age', 'Number of Diseases', 'Recent Hospitalization',
    'Number of Medications', 'Hour',
    'Creation to Assignment Interval',
    'Number of Previous Attendance',
    'Number of Previous Non-Attendance'
]

cat_cols = ['Sex', 'Insurance Type', 'Day', 'Month']

X = df[num_cols + cat_cols]
y = df[target]

# Preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', 'passthrough', cat_cols)
    ]
)

# Pipeline con SMOTE (usar ImbPipeline de imblearn)
pipeline = ImbPipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('select', SelectKBest(score_func=f_classif)),
    ('model', DecisionTreeClassifier(random_state=42))
])


param_grid = {
    'select__k': [10, 12, 'all'], 

    'model__max_depth': [8, 10, 12, None],
    'model__min_samples_split': [20, 50, 100],
    'model__min_samples_leaf': [10, 20, 40],
    'model__criterion': ['gini', 'entropy']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Iniciando GridSearchCV con SMOTE...")
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y)

print("\nMejores parámetros:")
print(grid.best_params_)

best_model = grid.best_estimator_

print("\n===== ENTRENAMIENTO REPETIDO =====")

N = 10

results = []

for seed in range(N):
    
    model_rep = ImbPipeline(steps=[
        ('preprocess', preprocessor),
        ('smote', SMOTE(random_state=seed)),
        ('select', SelectKBest(
            score_func=f_classif,
            k=grid.best_params_['select__k']
        )),
        ('model', DecisionTreeClassifier(
            random_state=seed,
            max_depth=grid.best_params_['model__max_depth'],
            min_samples_split=grid.best_params_['model__min_samples_split'],
            min_samples_leaf=grid.best_params_['model__min_samples_leaf'],
            criterion=grid.best_params_['model__criterion']
        ))
    ])
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    model_rep.fit(X_train, y_train)
    y_pred = model_rep.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    
    results.append((seed, acc, model_rep, y_test, y_pred))
    
    print(f"Seed {seed} → Accuracy: {acc:.4f}")

best_seed, best_acc, best_model_final, y_test_final, y_pred_final = max(
    results, key=lambda x: x[1]
)

print("\n" + "="*50)
print("MEJOR MODELO FINAL")
print("="*50)
print(f"Seed: {best_seed}")
print(f"Accuracy: {best_acc:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test_final, y_pred_final, digits=3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_final, y_pred_final))

# Mostrar distribución de clases antes y después de SMOTE
print("\n" + "="*50)
print("DISTRIBUCIÓN DE CLASES")
print("="*50)
print("\nOriginal:")
print(y.value_counts())
print(f"\nProporción: {y.value_counts(normalize=True)}")

Iniciando GridSearchCV con SMOTE...
Fitting 5 folds for each of 216 candidates, totalling 1080 fits

Mejores parámetros:
{'model__criterion': 'gini', 'model__max_depth': None, 'model__min_samples_leaf': 10, 'model__min_samples_split': 50, 'select__k': 12}

===== ENTRENAMIENTO REPETIDO =====
Seed 0 → Accuracy: 0.7659
Seed 1 → Accuracy: 0.7654
Seed 2 → Accuracy: 0.7705
Seed 3 → Accuracy: 0.7659
Seed 4 → Accuracy: 0.7654
Seed 5 → Accuracy: 0.7659
Seed 6 → Accuracy: 0.7743
Seed 7 → Accuracy: 0.7577
Seed 8 → Accuracy: 0.7776
Seed 9 → Accuracy: 0.7700

MEJOR MODELO FINAL
Seed: 8
Accuracy: 0.7776

Classification Report:

              precision    recall  f1-score   support

           0      0.829     0.835     0.832      2423
           1      0.676     0.668     0.672      1255

    accuracy                          0.778      3678
   macro avg      0.753     0.751     0.752      3678
weighted avg      0.777     0.778     0.777      3678


Confusion Matrix:
[[2022  401]
 [ 417  838]]

DIST

# Random Forest

In [10]:
df = pd.read_csv("../data/processed/dataset_desbalanceado.csv")

target = 'Appointment Type'

num_cols = [
    'Age', 'Number of Diseases', 'Recent Hospitalization',
    'Number of Medications', 'Hour',
    'Creation to Assignment Interval',
    'Number of Previous Attendance',
    'Number of Previous Non-Attendance'
]

cat_cols = ['Sex', 'Insurance Type', 'Day', 'Month']

X = df[num_cols + cat_cols]
y = df[target]

# Preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', 'passthrough', cat_cols)
    ]
)

# Pipeline con SMOTE y Random Forest
pipeline = ImbPipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('select', SelectKBest(score_func=f_classif)),   # 👈 NUEVO
    ('model', RandomForestClassifier(random_state=42))
])

# Grid de parámetros para Random Forest
param_grid = {
    'select__k': [10, 12, 'all'],   # 👈 NUEVO

    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 15, None],
    'model__min_samples_split': [10, 20],
    'model__min_samples_leaf': [5, 10],
    'model__max_features': ['sqrt', 'log2'],
    'model__criterion': ['gini', 'entropy']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Iniciando GridSearchCV con Random Forest y SMOTE...")
print(f"Total de combinaciones a probar: {3 * 4 * 3 * 3 * 2 * 2} candidatos")
print(f"Total de entrenamientos: {3 * 4 * 3 * 3 * 2 * 2 * 5} fits\n")

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y)

print("\n" + "="*60)
print("MEJORES PARÁMETROS ENCONTRADOS")
print("="*60)
for param, value in grid.best_params_.items():
    print(f"{param}: {value}")

print(f"\nMejor score en validación cruzada: {grid.best_score_:.4f}")

best_model = grid.best_estimator_

print("\n" + "="*60)
print("ENTRENAMIENTO REPETIDO CON DIFERENTES SEEDS")
print("="*60)

N = 4

results = []

for seed in range(N):
    
    model_rep = ImbPipeline(steps=[
        ('preprocess', preprocessor),
        ('smote', SMOTE(random_state=seed)),

        # 🔥 FEATURE SELECTION
        ('select', SelectKBest(
            score_func=f_classif,
            k=grid.best_params_['select__k']   # viene del GridSearch
        )),

        ('model', RandomForestClassifier(
            random_state=seed,
            n_estimators=grid.best_params_['model__n_estimators'],
            max_depth=grid.best_params_['model__max_depth'],
            min_samples_split=grid.best_params_['model__min_samples_split'],
            min_samples_leaf=grid.best_params_['model__min_samples_leaf'],
            max_features=grid.best_params_['model__max_features'],
            criterion=grid.best_params_['model__criterion']
        ))
    ])
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    model_rep.fit(X_train, y_train)
    y_pred = model_rep.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    
    results.append((seed, acc, model_rep, y_test, y_pred))
    
    print(f"Seed {seed:2d} → Accuracy: {acc:.4f}")


best_seed, best_acc, best_model_final, y_test_final, y_pred_final = max(
    results, key=lambda x: x[1]
)

print("\n" + "="*60)
print("MEJOR MODELO FINAL (RANDOM FOREST)")
print("="*60)
print(f"Seed: {best_seed}")
print(f"Accuracy: {best_acc:.4f}")

print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test_final, y_pred_final, digits=3))

print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)
print(confusion_matrix(y_test_final, y_pred_final))

# Importancia de características (Feature Importance)
print("\n" + "="*60)
print("IMPORTANCIA DE CARACTERÍSTICAS (TOP 10)")
print("="*60)

# Obtener el modelo RF entrenado del pipeline
rf_model = best_model_final.named_steps['model']
feature_names = num_cols + cat_cols

# Crear DataFrame con importancias
importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(importances.head(10).to_string(index=False))

# Estadísticas del entrenamiento repetido
print("\n" + "="*60)
print("ESTADÍSTICAS DE LOS 10 ENTRENAMIENTOS")
print("="*60)
accuracies = [r[1] for r in results]
print(f"Accuracy promedio: {np.mean(accuracies):.4f}")
print(f"Desviación estándar: {np.std(accuracies):.4f}")
print(f"Accuracy mínimo: {np.min(accuracies):.4f}")
print(f"Accuracy máximo: {np.max(accuracies):.4f}")

# Distribución de clases
print("\n" + "="*60)
print("DISTRIBUCIÓN DE CLASES")
print("="*60)
print("\nOriginal:")
print(y.value_counts())
print(f"\nProporción:")
print(y.value_counts(normalize=True).round(3))

Iniciando GridSearchCV con Random Forest y SMOTE...
Total de combinaciones a probar: 432 candidatos
Total de entrenamientos: 2160 fits

Fitting 5 folds for each of 288 candidates, totalling 1440 fits


KeyboardInterrupt: 

# SVM

In [6]:
pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('select', SelectKBest(score_func=f_classif)),
    ('model', SVC(probability=True, random_state=42))
])

param_grid = {
    'select__k': [10, 12, 'all'],

    'model__C': [0.1, 1, 10],          # Regularización
    'model__kernel': ['linear', 'rbf'], 
    'model__gamma': ['scale', 'auto']  # Solo afecta rbf
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='recall',
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y)

print("Mejores parámetros:", grid.best_params_)
best_model = grid.best_estimator_

results = []
N = 10
df
for seed in range(N):
    
    model_rep = ImbPipeline(steps=[
        ('preprocess', preprocessor),
        ('smote', SMOTE(random_state=seed)),
        ('select', SelectKBest(
            score_func=f_classif,
            k=grid.best_params_['select__k']
        )),
        ('model', SVC(
            probability=True,
            random_state=seed,
            C=grid.best_params_['model__C'],
            kernel=grid.best_params_['model__kernel'],
            gamma=grid.best_params_['model__gamma']
        ))
    ])
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    model_rep.fit(X_train, y_train)
    y_pred = model_rep.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    results.append((seed, acc, model_rep, y_test, y_pred))
    
    print(f"Seed {seed} → Accuracy: {acc:.4f}")

best_seed, best_acc, best_model_final, y_test_final, y_pred_final = max(
    results, key=lambda x: x[1]
)

print("\nMEJOR MODELO FINAL (SVM)")
print(f"Seed: {best_seed}")
print(f"Accuracy: {best_acc:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test_final, y_pred_final, digits=3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_final, y_pred_final))


Fitting 5 folds for each of 36 candidates, totalling 180 fits


KeyboardInterrupt: 